To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [19]:
from unsloth import FastLanguageModel  # FastVisionModel for LLMs
import torch
max_seq_length = 4096
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",  # Llama-3.1 2x faster
    "unsloth/Mistral-Small-Instruct-2409",  # Mistral 22b 2x faster!
    "unsloth/Phi-4",  # Phi-4 2x faster!
    "unsloth/Phi-4-unsloth-bnb-4bit",  # Phi-4 Unsloth Dynamic 4-bit Quant
    "unsloth/gemma-2-9b-bnb-4bit",  # Gemma 2x faster!
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # Qwen 2.5 2x faster!
    "unsloth/Llama-3.2-1B-bnb-4bit",  # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
]  # More models at https://unsloth.ai/docs/get-started/all-our-models

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-4",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [21]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Already have LoRA adapters! We shall skip this step.


<a name="Data"></a>
### Data Prep

In [22]:
from datasets import load_dataset
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}


### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Clean the input: Strip trailing whitespace which often causes token bleeding
        cleaned_input = input_text.strip()
        # Format: Note the double newline before Response
        text = alpaca_prompt.format(instruction, cleaned_input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }
# Split into 80% training and 20% validation
# shuffle=True ensures the different log files are mixed together
dataset = load_dataset("Jaiccc/model0_boundary_predict_streaming", split="train")
split_dataset = dataset.train_test_split(test_size=0.2, seed=42, shuffle=True)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)

Training samples: 1099
Validation samples: 275


In [23]:
print("=== THE RAW OUTPUT COLUMN ===")
print(train_dataset[1]['output'])

print("\n=== THE FORMATTED TEXT COLUMN (What the model actually sees) ===")
print(train_dataset[1]['text'])

=== THE RAW OUTPUT COLUMN ===
461.321713, old event

=== THE FORMATTED TEXT COLUMN (What the model actually sees) ===
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\n`), is just the completion of the input phase.
- 

In [24]:
def manual_labeling_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    all_input_ids = []
    all_labels = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # 1. Clean and truncate input as before
        cleaned_input = input_text.strip()
        if len(cleaned_input) > 3000:
            cleaned_input = cleaned_input[:1500] + "\n[TRUNCATED]\n" + cleaned_input[-1500:]

        # 2. Build the two parts separately
        prompt_part = f"### Instruction:\n{instruction}\n\n### Input:\n{cleaned_input}\n\n### Response: "
        response_part = f"{output}{tokenizer.eos_token}"

        # 3. Tokenize them separately
        prompt_ids = tokenizer.encode(prompt_part, add_special_tokens=True)
        response_ids = tokenizer.encode(response_part, add_special_tokens=False)

        # 4. Create the labels
        # Mask the prompt part with -100 (ignore)
        # Keep the response part as the actual token IDs (learn)
        input_ids = prompt_ids + response_ids
        labels = ([-100] * len(prompt_ids)) + response_ids

        # 5. Handle truncation to max_seq_length
        all_input_ids.append(input_ids[:max_seq_length])
        all_labels.append(labels[:max_seq_length])

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
    }

# Apply the manual labeling
# NOTE: We remove the "text" field to ensure the trainer uses our manual labels
train_dataset = train_dataset.map(manual_labeling_func, batched = True, remove_columns = train_dataset.column_names)

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [25]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    # IMPORTANT: We don't use dataset_text_field because we provided labels manually
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "outputs",
    ),
)

We verify masking is actually done:

In [26]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'### Instruction:\nYour task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".\n\n### DEFINITION OF A NEW EVENT:\n1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).\n2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from \'fai-mirror finished\' to \'Copying the nfsroot\').\n3. **Internal Logic:** Shifts from downloading to processing.\n\n### WHAT IS *NOT* A NEW EVENT (OLD EVENT):\n- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\\n`), is just the completion of the input phase.\n- **Incomplete Tasks:** Continuous system output without a clear phase shift.\n\nCRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract timestamps from the "### CONTEXT" section. Output only the timestamp

In [27]:
"""
By setting the labels for the instruction and input context to -100, the training engine is instructed
to calculate the loss only for the response tokens. This effectively masks out the prompt's influence
on weight updates, forcing the model to ignore the task instructions and focus entirely on learning
the precise pattern of predicting the timestamp and event classification.
"""
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

We can see the System and Instruction prompts are successfully masked!

In [28]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.251 GB.
33.631 GB of memory reserved.


In [29]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,099 | Num Epochs = 3 | Total steps = 414
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 65,536,000 of 14,725,043,200 (0.45% trained)


Step,Training Loss
1,0.135000
2,0.158600
3,0.263100
4,0.077300
5,0.004500
6,0.072700
7,0.029400
8,0.073600
9,0.038500
10,0.017100


wandb: WARNING URL not available in offline run


train/epoch,▁▁▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇███
train/grad_norm,█▆▅▃▆▂▇▂▁▁▂▁▃▂▁▂▄▂▂▂▁▁▁▂▁▁▁▁▁▁▃▃▁▁▁▁▁▁▂▁
train/learning_rate,▇████▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▁▁▁
train/loss,▅▁▂▁▃▁▁▃▁▁▂▂▁▁▂▂▂█▁▁▁▂▁▂▁▁▁▁▁▁▂▂▁▁▁▁▂▁▁▃
total_flos,2.8654500324409344e+17
train/epoch,3
train/global_step,414
train/grad_norm,0.00353
train/learning_rate,0.0
train/loss,0.0002


In [30]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1210.2363 seconds used for training.
20.17 minutes used for training.
Peak reserved memory = 44.523 GB.
Peak reserved memory for training = 10.892 GB.
Peak reserved memory % of max memory = 56.18 %.
Peak reserved memory for training % of max memory = 13.744 %.


Inference using the fine tunned model on validation set


In [31]:
import re
FastLanguageModel.for_inference(model)
# 1. Initialize counters for a quick summary
total_samples = len(val_dataset)
correct_count = 0

print(f"Scanning {total_samples} samples...\n")

for i in range(total_samples):
    sample = val_dataset[i]

    # 2. Extract components
    instruction = sample['instruction']
    input_data = sample['input']
    expected_output = sample['output'].strip()

    # 3. Format the prompt
    combined_user_text = f"{instruction}\n\n{input_data}"
    prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

    # 4. Generate
    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=64, # Classification is short, 64 is plenty
        use_cache=True,
        temperature=0.1,   # Lower temperature for more consistent classification
    )

    # 5. Decode and Clean
    raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

    # Extract only the assistant's specific response
    m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
    model_result = m.group(1).strip() if m else raw_output.split("assistant")[-1].strip()

    # 6. Compare and Print
    is_correct = "✅" if model_result == expected_output else "❌"
    if model_result == expected_output:
        correct_count += 1

    print(f"--- Sample {i} ---")
    print(f"EXPECTED: {expected_output}")
    print(f"MODEL:    {model_result} {is_correct}")
    print("-" * 20)

# 7. Final Summary
accuracy = (correct_count / total_samples) * 100
print(f"\nFinal Accuracy: {accuracy:.2f}% ({correct_count}/{total_samples})")

Scanning 275 samples...

--- Sample 0 ---
EXPECTED: 6.620943, new event
MODEL:    6.620943, new event ✅
--------------------
--- Sample 1 ---
EXPECTED: 2404.686829, old event
MODEL:    2404.686829, old event ✅
--------------------
--- Sample 2 ---
EXPECTED: 342.705528, new event
MODEL:    342.705528, new event ✅
--------------------
--- Sample 3 ---
EXPECTED: 863.730703, old event
MODEL:    863.730703, old event ✅
--------------------
--- Sample 4 ---
EXPECTED: 507.641025, old event
MODEL:    507.641025, old event ✅
--------------------
--- Sample 5 ---
EXPECTED: 1549.900047, old event
MODEL:    1549.900047, old event ✅
--------------------
--- Sample 6 ---
EXPECTED: 53.985281, new event
MODEL:    53.985281, new event ✅
--------------------
--- Sample 7 ---
EXPECTED: 2373.881574, new event
MODEL:    2373.881574, new event ✅
--------------------
--- Sample 8 ---
EXPECTED: 8709.088506, old event
MODEL:    8709.088506, old event ✅
--------------------
--- Sample 9 ---
EXPECTED: 1506.09532

In [32]:
# ==========================================
# SAFE SAVE AND PUSH TO HUGGING FACE
# ==========================================
from google.colab import userdata
from huggingface_hub import login

# 1. Pull the token safely from Colab Secrets
# This ensures your token never appears in the code or logs
hf_token = userdata.get('HF_TOKEN')

# 2. Log in to Hugging Face
login(token = hf_token)

# 3. Define your target repository
repo_name = "Jaiccc/model_0_streaming_timestamp"

print(f"Uploading model adapters to {repo_name}...")

# 4. Push the lightweight LoRA adapters
# private=True ensures the model is not public after upload
model.push_to_hub(repo_name, token = hf_token, private = True)
tokenizer.push_to_hub(repo_name, token = hf_token, private = True)

print("✅ Upload complete! The model is saved privately.")

Uploading model adapters to Jaiccc/model_0_streaming_timestamp...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 62.9kB /  262MB            

Saved model to https://huggingface.co/Jaiccc/model_0_streaming_timestamp


No files have been modified since last commit. Skipping to prevent empty commit.


✅ Upload complete! The model is saved privately.


Baseline model inference

In [17]:
# 1. Load the Base Model
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(base_model) # Enable 2x faster inference

# 2. Initialize counters
total_samples = len(val_dataset)
correct_count = 0

print(f"--- BASE MODEL EVALUATION ---")
print(f"Scanning {total_samples} samples from validation set...\n")

for i in range(total_samples):
    sample = val_dataset[i]

    # Extract components
    instruction = sample['instruction']
    input_data = sample['input']
    expected_output = sample['output'].strip()

    # Format the ChatML prompt for Phi-4
    combined_user_text = f"{instruction}\n\n{input_data}"
    prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

    # Generate using the base_model
    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    outputs = base_model.generate(
        input_ids = inputs,
        max_new_tokens = 64,
        use_cache = True,
        temperature = 0.1,
    )

    # Decode and Clean
    raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

    # Extract assistant response using regex
    m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
    if m:
        model_result = m.group(1).strip()
    else:
        # Fallback if the model doesn't generate the closing tag
        model_result = raw_output.split("assistant")[-1].replace("<|im_sep|>", "").strip()

    # Compare
    is_correct = "✅" if model_result == expected_output else "❌"
    if model_result == expected_output:
        correct_count += 1

    print(f"--- Sample {i} ---")
    print(f"EXPECTED: {expected_output}")
    print(f"MODEL:    {model_result} {is_correct}")
    print("-" * 20)

# 3. Final Summary
accuracy = (correct_count / total_samples) * 100
print(f"\n[BASELINE] Final Accuracy: {accuracy:.2f}% ({correct_count}/{total_samples})")

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- BASE MODEL EVALUATION ---
Scanning 275 samples from validation set...

--- Sample 0 ---
EXPECTED: 6.620943, new event
MODEL:    6.620943, new event ✅
--------------------
--- Sample 1 ---
EXPECTED: 2404.686829, old event
MODEL:    2404.686829, old event ✅
--------------------
--- Sample 2 ---
EXPECTED: 342.705528, new event
MODEL:    342.705528, old event ❌
--------------------
--- Sample 3 ---
EXPECTED: 863.730703, old event
MODEL:    863.730703, old event ✅
--------------------
--- Sample 4 ---
EXPECTED: 507.641025, old event
MODEL:    507.641025, old event ✅
--------------------
--- Sample 5 ---
EXPECTED: 1549.900047, old event
MODEL:    1549.900047, new event ❌
--------------------
--- Sample 6 ---
EXPECTED: 53.985281, new event
MODEL:    53.985281, old event ❌
--------------------
--- Sample 7 ---
EXPECTED: 2373.881574, new event
MODEL:    2373.881574, new event ✅
--------------------
--- Sample 8 ---
EXPECTED: 8709.088506, old event
MODEL:    8709.088506, old event ✅
--------